# 02 - Modelos clasicos y baseline de reglas

Entrena y evalua los modelos vistos en el curso sobre el corpus sintetico:
**Naive Bayes**, **KNN**, **SVM (lineal y RBF)**, **Arbol de Decision**,
**Random Forest** y **Perceptron multicapa (MLP)**; ademas de un **baseline de
reglas** por palabras clave. Se reporta F1-macro, exactitud, desempeno por
idioma, latencia y la matriz de confusion.


In [1]:
import sys, os
sys.path.insert(0, '..')
import importlib, experimentos as ex
importlib.reload(ex)
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


## 1. Carga, preprocesamiento y particion estratificada

El preprocesamiento normaliza Unicode, pasa a minusculas y elimina URLs,
**conservando los emojis** como tokens. La representacion es TF-IDF de palabras
(1-2 gramas) unida a TF-IDF de caracteres (3-5 gramas) mediante `FeatureUnion`.
Se reserva un 15% como test con muestreo estratificado por interseccion
intencion x idioma.

In [2]:
df = ex.cargar()
X, y, lang = df['text'].values, df['intent'].values, df['lang'].values
strat = np.array([f'{a}|{b}' for a,b in zip(y,lang)])
idx = np.arange(len(X))
itr, ite = train_test_split(idx, test_size=0.15, random_state=ex.RANDOM_STATE, stratify=strat)
Xtr,Xte,ytr,yte,lang_te = X[itr],X[ite],y[itr],y[ite],lang[ite]
print(f'Train={len(Xtr)}  Test={len(Xte)}')


Train=1122  Test=198


## 2. Baseline de reglas

Un clasificador determinista de expresiones regulares por intencion. Establece
el piso contra el cual se comparan los modelos supervisados.

In [3]:
ypred_r = np.array([ex.clasificar_reglas(t) for t in Xte])
from sklearn.metrics import accuracy_score, f1_score
print(f"Reglas -> acc={accuracy_score(yte,ypred_r):.4f}  F1-macro={f1_score(yte,ypred_r,average='macro'):.4f}")


Reglas -> acc=0.8030  F1-macro=0.8251


## 3. Entrenamiento y evaluacion de los modelos clasicos

In [4]:
filas=[]; evals={}
for nombre, modelo in ex.construir_modelos().items():
    r = ex.evaluar_modelo(nombre, modelo, Xtr, ytr, Xte, yte, lang_te)
    evals[nombre]=r
    filas.append({'modelo':nombre,'acc':r['acc'],'f1_macro':r['f1_macro'],
                  'lat_ms':round(r['lat_ms'],3),'fit_s':round(r['t_fit_s'],2)})
tabla = pd.DataFrame(filas).sort_values('f1_macro', ascending=False).reset_index(drop=True)
tabla


,modelo,acc,f1_macro,lat_ms,fit_s
0,Naive Bayes,0.964646,0.964911,0.024,0.03
1,MLP,0.964646,0.964911,0.031,0.98
2,SVM lineal,0.959596,0.960262,0.040,0.14
3,SVM RBF,0.944444,0.945140,0.280,3.41
4,Random Forest,0.934343,0.934370,0.161,0.28
5,KNN,0.924242,0.923880,0.042,0.03
6,Arbol de Decision,0.828283,0.826758,0.025,0.09


## 4. Comparacion visual (F1-macro) frente a los baselines

In [5]:
LLM = ex.LLM
fig, ax = plt.subplots(figsize=(7.5,4))
t = tabla.sort_values('f1_macro')
ax.barh(t['modelo'], t['f1_macro'], color='#3b6fb0')
ax.axvline(f1_score(yte,ypred_r,average='macro'), color='#c0392b', ls='--', label='Reglas')
ax.axvline(LLM['f1_macro'], color='#27ae60', ls='--', label=f"LLM ({LLM['f1_macro']:.3f})")
ax.set_xlim(0.7,1.02); ax.set_xlabel('F1-macro'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()


## 5. Desempeno por idioma del mejor modelo

In [6]:
mejor = max(evals.values(), key=lambda r:r['f1_macro'])
print('Mejor modelo:', mejor['nombre'])
pd.DataFrame(mejor['por_idioma']).T


Mejor modelo: Naive Bayes


,acc,f1,n
es,0.956522,0.956624,138.0
en,0.972222,0.971429,36.0
pt,1.000000,1.000000,24.0


## 6. Matriz de confusion y reporte por clase del mejor modelo

Los errores se concentran en los dos pares lexicamente ambiguos:
`consultar_medicina` ↔ `agendar_cita_medica` y `consulta_banca` ↔
`soporte_tecnico`, exactamente los patrones anticipados en el primer informe.

In [7]:
labels = ex.INTENT_ORDER
cm = confusion_matrix(yte, mejor['ypred'], labels=labels)
fig, ax = plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
plt.title(f"Matriz de confusion - {mejor['nombre']}"); plt.tight_layout(); plt.show()
print(classification_report(yte, mejor['ypred'], labels=labels, zero_division=0))


                     precision    recall  f1-score   support

 constituir_empresa       1.00      1.00      1.00        16
      comprar_pizza       1.00      1.00      1.00        17
     reservar_hotel       1.00      1.00      1.00        16
      comprar_vuelo       1.00      1.00      1.00        17
 consultar_medicina       1.00      0.76      0.87        17
agendar_cita_medica       0.80      1.00      0.89        16
     consulta_banca       0.94      0.88      0.91        17
    soporte_tecnico       0.89      0.94      0.91        17
            reclamo       1.00      1.00      1.00        17
 seguimiento_pedido       1.00      1.00      1.00        16
             saludo       1.00      1.00      1.00        16
          despedida       1.00      1.00      1.00        16

           accuracy                           0.96       198
          macro avg       0.97      0.97      0.96       198
       weighted avg       0.97      0.96      0.96       198

